#  Network Analysis

In [ ]:
# ==========================================================
# Cell 1 : Import Libraries
# ==========================================================
# Project : TradeGraph
# Task    : Graph Analysis
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import random
import numpy as np
import pandas as pd

# Graph Analysis
import networkx as nx

# Visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ==========================================================
# Reproducibility
# ==========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Libraries successfully imported.")

In [ ]:
# ==========================================================
# Cell 2 : Load Cleaned Dataset
# ==========================================================

df = pd.read_csv(
    "International_Trade_Semiconductor_Cleaned_Data_1995_2024.csv"
)

print("="*60)
print("Dataset Successfully Loaded")
print("="*60)

print(f"Observations : {len(df):,}")
print(f"Variables    : {df.shape[1]}")

print("\nYears")

print(sorted(df["year"].unique()))

print("\nFirst five observations")

display(df.head())

In [ ]:
# ==========================================================
# Cell 3 : Graph Preprocessing
# ==========================================================

print("="*60)
print("Graph Preprocessing")
print("="*60)

edge_features = (
    df
    .groupby(
        [
            "year",
            "exporter",
            "importer"
        ]
    )
    .agg(

        total_trade_value=("trade_value","sum"),

        average_trade_value=("trade_value","mean"),

        traded_products=("product_code","nunique"),

        dominant_product=(
            "product",
            lambda x: x.value_counts().index[0]
        ),

        diversification_index=(
            "product_code",
            lambda x: x.nunique()
        )

    )
    .reset_index()
)

print(edge_features.head())

In [ ]:
# ==========================================================
# Cell 4 : Construction of Yearly Trade Graphs
# ==========================================================

graphs = {}

years = sorted(edge_features["year"].unique())

for year in years:

    yearly_data = edge_features[
        edge_features["year"] == year
    ]

    G = nx.DiGraph(year=year)

    for _, row in yearly_data.iterrows():

        G.add_edge(

            row["exporter"],

            row["importer"],

            weight=row["total_trade_value"],

            average_trade=row["average_trade_value"],

            products=row["traded_products"],

            diversification=row["diversification_index"],

            dominant_product=row["dominant_product"]

        )

    graphs[year] = G

print("="*60)

for year, graph in graphs.items():

    print(

        f"Year {year} : "

        f"{graph.number_of_nodes()} nodes | "

        f"{graph.number_of_edges()} edges"

    )

In [ ]:
# ==========================================================
# Cell 5 : Graph Statistical Analysis
# ==========================================================

print("=" * 70)
print("Graph Statistical Analysis")
print("=" * 70)

graph_statistics = []

for year, G in graphs.items():

    stats = {

        "Year": year,

        "Nodes": G.number_of_nodes(),

        "Edges": G.number_of_edges(),

        "Density": nx.density(G),

        "Average Degree":
            np.mean(
                [d for _, d in G.degree()]
            ),

        "Average Clustering":
            nx.average_clustering(
                G.to_undirected()
            ),

        "Connected Components":
            nx.number_connected_components(
                G.to_undirected()
            )

    }

    graph_statistics.append(stats)

graph_statistics = pd.DataFrame(graph_statistics)

# Round all decimal columns to 3 decimal places
graph_statistics = graph_statistics.round(3)

display(graph_statistics)

In [ ]:
# ==========================================================
# Cell 5.1 : Trade Network Analysis
# ==========================================================

print("=" * 70)
print("Trade Network Analysis")
print("=" * 70)

for year in years:

    print("\n")
    print("=" * 70)
    print(f"YEAR : {year}")
    print("=" * 70)

    yearly = edge_features[
        edge_features["year"] == year
    ].copy()

    # ------------------------------------------------------
    # Top Exporters
    # ------------------------------------------------------

    top_exporters = (
        yearly
        .groupby("exporter")["total_trade_value"]
        .sum()
        .div(1_000_000)                       # Convert to Billion USD, BACI dataset had already been in thousand usd dollars
        .round(3)                             # 3 decimals
        .sort_values(ascending=False)
        .head(25)
        .rename("Trade Value (Billion USD)")
    )

    print("\nTop 25 Exporters")
    display(top_exporters)

    # ------------------------------------------------------
    # Top Importers
    # ------------------------------------------------------

    top_importers = (
        yearly
        .groupby("importer")["total_trade_value"]
        .sum()
        .div(1_000_000)
        .round(3)
        .sort_values(ascending=False)
        .head(25)
        .rename("Trade Value (Billion USD)")
    )

    print("\nTop 25 Importers")
    display(top_importers)

    # ------------------------------------------------------
    # Top Bilateral Trade Flows
    # ------------------------------------------------------

    top_flows = (
        yearly[
            [
                "exporter",
                "importer",
                "total_trade_value"
            ]
        ]
        .assign(
            **{
                "Trade Value (Billion USD)":
                lambda x: (
                    x["total_trade_value"] / 1_000_000
                ).round(3)
            }
        )
        .drop(columns="total_trade_value")
        .sort_values(
            by="Trade Value (Billion USD)",
            ascending=False
        )
        .head(25)
    )

    print("\nTop 25 Bilateral Trade Flows")
    display(top_flows)

    # ------------------------------------------------------
    # Edge Weight Distribution
    # ------------------------------------------------------

    plt.figure(figsize=(7,5))

    plt.hist(
        yearly["total_trade_value"] / 1_000_000,
        bins=50
    )

    plt.xlabel("Trade Value (Billion USD)")

    plt.ylabel("Frequency")

    plt.title(
        f"Trade Value Distribution ({year})"
    )

    plt.show()

    # ------------------------------------------------------
    # Network Centrality
    # ------------------------------------------------------

    G = graphs[year]

    pagerank = nx.pagerank(
        G,
        weight="weight"
    )

    degree = dict(G.degree())

    centrality = pd.DataFrame({

        "Country": list(G.nodes()),

        "Degree":
            [degree[n] for n in G.nodes()],

        "PageRank":
            [round(pagerank[n], 3) for n in G.nodes()]

    })

    centrality = centrality.sort_values(
        "PageRank",
        ascending=False
    )

    print("\nTop 25 Most Central Countries")

    display(
        centrality.head(25)
    )

In [ ]:
# ==========================================================
# Cell 5.2 : Network Centrality Analysis
# ==========================================================

print("=" * 70)
print("Network Centrality Analysis")
print("=" * 70)

centrality_results = {}

for year, G in graphs.items():

    print("\n")
    print("=" * 70)
    print(f"YEAR : {year}")
    print("=" * 70)

    # ------------------------------------------------------
    # Centrality Measures
    # ------------------------------------------------------

    degree = nx.degree_centrality(G)

    betweenness = nx.betweenness_centrality(
        G,
        normalized=True
    )

    closeness = nx.closeness_centrality(G)

    pagerank = nx.pagerank(
        G,
        weight="weight"
    )

    eigenvector = nx.eigenvector_centrality(
        G,
        max_iter=500
    )

    centrality = pd.DataFrame({

        "Country": list(G.nodes()),

        "Degree Centrality":
            [degree[n] for n in G.nodes()],

        "Betweenness":
            [betweenness[n] for n in G.nodes()],

        "Closeness":
            [closeness[n] for n in G.nodes()],

        "Eigenvector":
            [eigenvector[n] for n in G.nodes()],

        "PageRank":
            [pagerank[n] for n in G.nodes()]

    })

    # Sort by PageRank
    centrality = centrality.sort_values(
        by="PageRank",
        ascending=False
    )

    # Round all decimal columns to 3 decimals
    centrality = centrality.round(3)

    # Save results
    centrality_results[year] = centrality

    print("Top 25 Most Influential Countries")

    display(
        centrality.head(25)
    )

In [ ]:
# Cell 5.3 : Community Detection

from networkx.algorithms.community import louvain_communities

print("=" * 70)
print("Community Detection")
print("=" * 70)

community_statistics = []

communities_results = {}

for year, G in graphs.items():

    print("\n")
    print("=" * 70)
    print(f"YEAR : {year}")
    print("=" * 70)

    G_undirected = G.to_undirected()

    communities = louvain_communities(
        G_undirected,
        weight="weight",
        seed=42
    )

    modularity = nx.community.modularity(
        G_undirected,
        communities,
        weight="weight"
    )

    communities_results[year] = communities

    sizes = [len(c) for c in communities]

    stats = {

        "Year": year,

        "Number of Communities":
            len(communities),

        "Largest Community":
            max(sizes),

        "Smallest Community":
            min(sizes),

        "Average Community Size":
            np.mean(sizes),

        "Modularity":
            modularity

    }

    community_statistics.append(stats)

    print(f"Detected Communities : {len(communities)}")

    print(f"Modularity Score     : {modularity:.4f}")

community_statistics = pd.DataFrame(
    community_statistics
)

print("\n")

display(
    community_statistics
)

In [ ]:
# ==========================================================
# Cell 5.4 : Temporal Evolution of the Trade Network
# ==========================================================

print("=" * 70)
print("Temporal Evolution of the Trade Network")
print("=" * 70)

temporal_results = graph_statistics.merge(

    community_statistics,

    on="Year"

)

# Round all decimal columns to 3 decimals
temporal_results = temporal_results.round(3)

display(
    temporal_results
)

# ----------------------------------------------------------
# Evolution plots
# ----------------------------------------------------------

metrics = [

    "Nodes",

    "Edges",

    "Density",

    "Average Degree",

    "Average Clustering",

    "Modularity",

    "Number of Communities"

]

for metric in metrics:

    plt.figure(figsize=(7,4))

    plt.plot(

        temporal_results["Year"],

        temporal_results[metric],

        marker="o",

        linewidth=2

    )

    plt.title(metric)

    plt.xlabel("Year")

    plt.ylabel(metric)

    plt.grid(False)

    plt.tight_layout()

    plt.show()

In [ ]:
# ============================
# Save temporal_results 
# ============================

filename_csv = "InternationalTrade_SemiConductor_Temporal Evolution of the Trade Network_1995_2024.csv"

temporal_results.to_csv(filename_csv, index=False)

print(f"World Semiconductor temporal_results successfully saved as:\n{filename_csv}")

In [ ]:
# ==========================================================
# Cell 5.5 : Evolution of Global Semiconductor Trade
# ==========================================================

print("=" * 70)
print("Evolution of Global Semiconductor Trade")
print("=" * 70)

# ----------------------------------------------------------
# Annual Global Trade (Million USD)
# ----------------------------------------------------------

trade_evolution = (
    df
    .groupby("year")["trade_value"]
    .sum()
    .div(1_000_000)          # Convert to Million USD
    .reset_index()
)

trade_evolution.columns = [
    "Year",
    "Total Trade (Billion USD)"
]

# ----------------------------------------------------------
# Annual Growth Rate (%)
# ----------------------------------------------------------

trade_evolution["Annual Growth Rate (%)"] = (
    trade_evolution["Total Trade (Billion USD)"]
    .pct_change()
    * 100
)

# Round all decimal values
trade_evolution = trade_evolution.round(3)

display(trade_evolution)

# ----------------------------------------------------------
# Figure 1 : Evolution of Global Semiconductor Trade
# ----------------------------------------------------------

plt.figure(figsize=(10,5))

plt.plot(
    trade_evolution["Year"],
    trade_evolution["Total Trade (Billion USD)"],
    marker="o",
    linewidth=2
)

plt.title("Evolution of Global Semiconductor Trade")

plt.xlabel("Year")

plt.ylabel("Billion USD")

plt.grid(False)

plt.tight_layout()

plt.show()

# ----------------------------------------------------------
# Figure 2 : Annual Growth Rate
# ----------------------------------------------------------

plt.figure(figsize=(10,5))

plt.bar(
    trade_evolution["Year"],
    trade_evolution["Annual Growth Rate (%)"],
    width=0.8
)

plt.axhline(
    y=0,
    color="black",
    linewidth=1
)

plt.title("Annual Growth Rate of Global Semiconductor Trade")

plt.xlabel("Year")

plt.ylabel("Growth Rate (%)")

plt.grid(False)

plt.tight_layout()

plt.show()

In [ ]:
# ============================
# Save trade_evolution 
# ============================

filename_csv = "InternationalTrade_SemiConductor_TradeEvolution_1995_2024.csv"

trade_evolution.to_csv(filename_csv, index=False)

print(f"World Semiconductor Trade evolution successfully saved as:\n{filename_csv}")

In [ ]:
# ==========================================================
# Cell 7 : Node Feature Engineering
# ==========================================================

node_features = {}

for year, G in graphs.items():

    features = []

    nodes = []

    # pagerank = nx.pagerank(G)
    pagerank = nx.pagerank(
        G,
        weight="weight"
    )

    betweenness = nx.betweenness_centrality(G)

    clustering = nx.clustering(
        G.to_undirected()
    )

    for node in G.nodes():

        nodes.append(node)

        features.append([

            G.in_degree(node),

            G.out_degree(node),

            G.degree(node),

            pagerank[node],

            betweenness[node],

            clustering[node]

        ])

    features = np.array(features)

    scaler = StandardScaler()

    features = scaler.fit_transform(features)

    node_features[year] = {

        "nodes": nodes,

        "features": features

    }

print("Node features successfully created.")

In [ ]:
# ==========================================================
# Cell 8 : Edge Feature Engineering
# ==========================================================

edge_vectors = {}

for year in years:

    yearly = edge_features[
        edge_features["year"] == year
    ]

    edge_vectors[year] = yearly[[

        "exporter",

        "importer",

        "total_trade_value",

        "average_trade_value",

        "traded_products",

        "diversification_index"

    ]]

print("Edge features successfully created.")

In [ ]:
# ==========================================================
# Cell 9 : Export Trade Networks to Gephi
# ==========================================================

import os
import networkx as nx
from networkx.algorithms.community import louvain_communities

# ----------------------------------------------------------
# Output folder
# ----------------------------------------------------------

output_folder = "Gephi_Graphs"

os.makedirs(output_folder, exist_ok=True)

print("=" * 70)
print("Exporting Trade Networks for Gephi")
print("=" * 70)

# ----------------------------------------------------------
# Number of main trade partners to keep
# ----------------------------------------------------------

TOP_FLOW_PER_COUNTRY = 5

for year in years:

    print(f"\nProcessing {year}...")

    ########################################################
    # Complete graph
    ########################################################

    G = graphs[year].copy()

    ########################################################
    # Node statistics
    ########################################################

    pagerank = nx.pagerank(
        G,
        weight="weight"
    )

    degree = dict(G.degree())

    degree_centrality = nx.degree_centrality(G)

    betweenness = nx.betweenness_centrality(
        G,
        normalized=True
    )

    closeness = nx.closeness_centrality(G)

    eigenvector = nx.eigenvector_centrality(
        G,
        max_iter=500
    )

    ########################################################
    # Community Detection (Louvain)
    ########################################################

    communities = louvain_communities(
        G.to_undirected(),
        weight="weight",
        seed=42
    )

    community_dict = {}

    for cid, community in enumerate(communities):

        for node in community:

            community_dict[node] = cid

    ########################################################
    # Add node attributes
    ########################################################

    for node in G.nodes():

        G.nodes[node]["Degree"] = degree[node]

        G.nodes[node]["DegreeCentrality"] = degree_centrality[node]

        G.nodes[node]["PageRank"] = pagerank[node]

        G.nodes[node]["Betweenness"] = betweenness[node]

        G.nodes[node]["Closeness"] = closeness[node]

        G.nodes[node]["Eigenvector"] = eigenvector[node]

        G.nodes[node]["Community"] = community_dict[node]

    ########################################################
    # Export complete graph
    ########################################################

    filename = os.path.join(
        output_folder,
        f"Semiconductor_Trade_version2_{year}.gexf"
    )

    nx.write_gexf(G, filename)

    ########################################################
    # Top 25 countries (Weighted PageRank)
    ########################################################

    top25 = sorted(
        pagerank,
        key=pagerank.get,
        reverse=True
    )[:25]

    G_top = G.subgraph(top25).copy()

    ########################################################
    # Export Top25
    ########################################################

    filename = os.path.join(
        output_folder,
        f"Semiconductor_Trade_version2_{year}_Top25.gexf"
    )

    nx.write_gexf(G_top, filename)

    ########################################################
    # Article Graph
    ########################################################

    G_article = nx.DiGraph()

    ########################################################
    # Add Top25 countries
    ########################################################

    G_article.add_nodes_from(top25)

    ########################################################
    # Add Top Incoming / Outgoing Trade Flows
    ########################################################

    for country in top25:

        # ----------------------------------------------
        # Top exports
        # ----------------------------------------------

        outgoing_edges = sorted(

            G.out_edges(country, data=True),

            key=lambda x: x[2]["weight"],

            reverse=True

        )[:TOP_FLOW_PER_COUNTRY]

        # ----------------------------------------------
        # Top imports
        # ----------------------------------------------

        incoming_edges = sorted(

            G.in_edges(country, data=True),

            key=lambda x: x[2]["weight"],

            reverse=True

        )[:TOP_FLOW_PER_COUNTRY]

        # ----------------------------------------------
        # Add outgoing edges
        # ----------------------------------------------

        for u, v, data in outgoing_edges:

            G_article.add_edge(
                u,
                v,
                **data
            )

        # ----------------------------------------------
        # Add incoming edges
        # ----------------------------------------------

        for u, v, data in incoming_edges:

            G_article.add_edge(
                u,
                v,
                **data
            )

    ########################################################
    # Copy node attributes
    ########################################################

    for node in G_article.nodes():

        if node in G.nodes():

            G_article.nodes[node].update(
                G.nodes[node]
            )

    ########################################################
    # Copy edge attributes
    ########################################################

    for u, v in G_article.edges():

        if G.has_edge(u, v):

            G_article[u][v].update(
                G[u][v]
            )

    ########################################################
    # Export Article Graph
    ########################################################

    filename = os.path.join(

        output_folder,

        f"Semiconductor_Trade_version2_{year}_Article.gexf"

    )

    nx.write_gexf(

        G_article,

        filename

    )

    ########################################################
    # Display summary
    ########################################################

    print(f"✓ {year} exported successfully.")

    print(
        f"   Complete Graph : "
        f"{G.number_of_nodes()} nodes | "
        f"{G.number_of_edges()} edges"
    )

    print(
        f"   Top25 Graph    : "
        f"{G_top.number_of_nodes()} nodes | "
        f"{G_top.number_of_edges()} edges"
    )

    print(
        f"   Article Graph  : "
        f"{G_article.number_of_nodes()} nodes | "
        f"{G_article.number_of_edges()} edges"
    )

print("\n")

print("=" * 70)
print("All graphs exported successfully.")
print("=" * 70)